In [4]:
import pandas as pd
import numpy as np

# Load raw data
incidents = pd.read_csv('../data/incidents_master.csv')
financial = pd.read_csv('../data/financial_impact.csv')
market = pd.read_csv('../data/market_impact.csv')

# Quick sanity check
print("incidents:", incidents.shape)
print("financial:", financial.shape)
print("market:", market.shape)

incidents: (850, 32)
financial: (778, 19)
market: (358, 31)


In [5]:
print("=== INCIDENTS ===")
print(list(incidents.columns))

print("\n=== FINANCIAL ===")
print(list(financial.columns))

print("\n=== MARKET ===")
print(list(market.columns))

=== INCIDENTS ===
['incident_id', 'company_name', 'company_revenue_usd', 'country_hq', 'industry_primary', 'industry_secondary', 'employee_count', 'is_public_company', 'stock_ticker', 'incident_date', 'incident_date_estimated', 'discovery_date', 'disclosure_date', 'attack_vector_primary', 'attack_vector_secondary', 'attack_chain', 'attributed_group', 'attribution_confidence', 'data_compromised_records', 'data_type', 'systems_affected', 'downtime_hours', 'data_source_primary', 'data_source_secondary', 'data_source_type', 'confidence_tier', 'quality_score', 'quality_grade', 'review_flag', 'notes', 'created_at', 'updated_at']

=== FINANCIAL ===
['incident_id', 'direct_loss_usd', 'direct_loss_method', 'ransom_demanded_usd', 'ransom_paid_usd', 'ransom_source', 'recovery_cost_usd', 'legal_fees_usd', 'regulatory_fine_usd', 'insurance_payout_usd', 'total_loss_usd', 'total_loss_method', 'total_loss_lower_bound', 'total_loss_upper_bound', 'inflation_adjusted_usd', 'cpi_index_used', 'notes', 'cre

In [6]:
print(incidents['industry_primary'].value_counts().sort_index())
print("\nunique values:", sorted(incidents['industry_primary'].unique()))

industry_primary
11         4
21        21
22        38
23        13
31-33     79
42        10
44-45     81
48-49     23
51       127
52       128
53         8
54        36
55         2
56        16
61        37
62       142
71        14
72        16
81         3
92        52
Name: count, dtype: int64

unique values: ['11', '21', '22', '23', '31-33', '42', '44-45', '48-49', '51', '52', '53', '54', '55', '56', '61', '62', '71', '72', '81', '92']


In [7]:
naics_map = {
    '11': 'Agriculture',
    '21': 'Mining & Energy',
    '22': 'Utilities',
    '23': 'Construction',
    '31-33': 'Manufacturing',
    '42': 'Wholesale Trade',
    '44-45': 'Retail Trade',
    '48-49': 'Transportation',
    '51': 'Information & Tech',
    '52': 'Finance & Insurance',
    '53': 'Real Estate',
    '54': 'Professional Services',
    '55': 'Management',
    '56': 'Administrative Services',
    '61': 'Education',
    '62': 'Healthcare',
    '71': 'Arts & Entertainment',
    '72': 'Hospitality',
    '81': 'Other Services',
    '92': 'Public Administration'
}

incidents['industry_name'] = incidents['industry_primary'].map(naics_map)

# Verify
print(incidents[['industry_primary', 'industry_name']].drop_duplicates().sort_values('industry_primary'))

    industry_primary            industry_name
253               11              Agriculture
118               21          Mining & Energy
28                22                Utilities
27                23             Construction
5              31-33            Manufacturing
124               42          Wholesale Trade
3              44-45             Retail Trade
19             48-49           Transportation
1                 51       Information & Tech
0                 52      Finance & Insurance
57                53              Real Estate
20                54    Professional Services
382               55               Management
30                56  Administrative Services
33                61                Education
22                62               Healthcare
167               71     Arts & Entertainment
89                72              Hospitality
554               81           Other Services
17                92    Public Administration


In [8]:
# Merge incidents with financial data
df = incidents.merge(financial, on='incident_id', how='left', suffixes=('', '_fin'))

# Merge with market data
df = df.merge(market, on='incident_id', how='left', suffixes=('', '_mkt'))

print("Merged shape:", df.shape)
print("\nMissing total_loss_usd:", df['total_loss_usd'].isna().sum())
print("Missing market data (car_0_to_30):", df['car_0_to_30'].isna().sum())

Merged shape: (850, 81)

Missing total_loss_usd: 72
Missing market data (car_0_to_30): 492


In [9]:
# Drop redundant columns from merges
cols_to_drop = [
    'stock_ticker_mkt',  # duplicate of stock_ticker
    'notes_fin',         # duplicate of notes
    'notes_mkt',         # duplicate of notes
    'created_at_fin', 'updated_at_fin',
    'created_at_mkt', 'updated_at_mkt',
    'created_at', 'updated_at'
]

# Only drop columns that actually exist
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)

# Convert incident_date to datetime
df['incident_date'] = pd.to_datetime(df['incident_date'], errors='coerce')
df['disclosure_date'] = pd.to_datetime(df['disclosure_date'], errors='coerce')
df['discovery_date'] = pd.to_datetime(df['discovery_date'], errors='coerce')

# Extract year for trend analysis
df['incident_year'] = df['incident_date'].dt.year

print("Final shape:", df.shape)
print("\nDate range:", df['incident_year'].min(), "to", df['incident_year'].max())
print("\nColumns:", list(df.columns))

Final shape: (850, 73)

Date range: 2021 to 2025

Columns: ['incident_id', 'company_name', 'company_revenue_usd', 'country_hq', 'industry_primary', 'industry_secondary', 'employee_count', 'is_public_company', 'stock_ticker', 'incident_date', 'incident_date_estimated', 'discovery_date', 'disclosure_date', 'attack_vector_primary', 'attack_vector_secondary', 'attack_chain', 'attributed_group', 'attribution_confidence', 'data_compromised_records', 'data_type', 'systems_affected', 'downtime_hours', 'data_source_primary', 'data_source_secondary', 'data_source_type', 'confidence_tier', 'quality_score', 'quality_grade', 'review_flag', 'notes', 'industry_name', 'direct_loss_usd', 'direct_loss_method', 'ransom_demanded_usd', 'ransom_paid_usd', 'ransom_source', 'recovery_cost_usd', 'legal_fees_usd', 'regulatory_fine_usd', 'insurance_payout_usd', 'total_loss_usd', 'total_loss_method', 'total_loss_lower_bound', 'total_loss_upper_bound', 'inflation_adjusted_usd', 'cpi_index_used', 'price_7d_before',

In [10]:
df.to_csv('../data/processed_data.csv', index=False)
print("Saved to data/processed_data.csv")
print("Shape:", df.shape)

# Final summary
print("\n--- DATASET SUMMARY ---")
print(f"Total incidents: {len(df)}")
print(f"Date range: {df['incident_year'].min()} - {df['incident_year'].max()}")
print(f"Industries: {df['industry_name'].nunique()}")
print(f"Countries: {df['country_hq'].nunique()}")
print(f"Attack vectors: {df['attack_vector_primary'].nunique()}")
print(f"Incidents with cost data: {df['total_loss_usd'].notna().sum()}")
print(f"Incidents with market data: {df['car_0_to_30'].notna().sum()}")
print(f"Avg total loss (USD): ${df['total_loss_usd'].mean():,.0f}")
print(f"Max total loss (USD): ${df['total_loss_usd'].max():,.0f}")

Saved to data/processed_data.csv
Shape: (850, 73)

--- DATASET SUMMARY ---
Total incidents: 850
Date range: 2021 - 2025
Industries: 20
Countries: 38
Attack vectors: 9
Incidents with cost data: 778
Incidents with market data: 358
Avg total loss (USD): $70,996,000
Max total loss (USD): $3,451,547,720
